In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as st
import yaml
import functions
%matplotlib inline

try:
    with open("../config.yaml", "r") as file:
        config = yaml.safe_load(file)
except:
    print("Yaml configuration file not found!")

In [2]:
df_stat = pd.read_csv("../data/clean/df_footprints_transformed_client_merged.csv")
df_stat = df_stat.drop("treament_group", axis=1)

In [ ]:
df_stat[df_stat["client_id"] ==9971962]

In [4]:
df_eda = pd.read_csv("../data/clean/df_footprints_transformed_client_merged.csv")
df_eda = df_eda.drop("treament_group", axis=1)
# df_eda.sort_values(by="time_diff",ascending= False)

In [5]:
df_eda["client_age_group"] = df_eda["client_age"].apply(functions.map_group_age)
df_eda

,client_id,visitor_id,visit_id,process_step,date_time,treatment_group,time_diff,time_diff_relative,time_diff_percent,client_tenure_year,client_tenure_month,client_age,gender,num_accts,balance,calls_6_mnth,logons_6_mnth,client_age_group
0,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test,0.0,0.0,0.00,3,46,29,U,2,25454.66,2,6,young
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test,7.0,7.0,0.04,3,46,29,U,2,25454.66,2,6,young
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test,39.0,32.0,0.25,3,46,29,U,2,25454.66,2,6,young
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test,138.0,99.0,0.87,3,46,29,U,2,25454.66,2,6,young
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test,158.0,20.0,1.00,3,46,29,U,2,25454.66,2,6,young
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305734,9999729,834634258_21862004160,870243567_56915814033_814203,step_2,2017-05-08 16:08:40,Test,15.0,10.0,0.20,10,124,31,F,3,107059.74,6,9,young
305735,9999729,834634258_21862004160,870243567_56915814033_814203,step_3,2017-05-08 16:09:19,Test,54.0,39.0,0.72,10,124,31,F,3,107059.74,6,9,young
305736,9999729,834634258_21862004160,870243567_56915814033_814203,confirm,2017-05-08 16:09:40,Test,75.0,21.0,1.00,10,124,31,F,3,107059.74,6,9,young
305737,9999832,145538019_54444341400,472154369_16714624241_585315,start,2017-05-16 16:46:03,Test,0.0,0.0,0.00,23,281,49,F,2,431887.61,1,4,middle-aged


In [ ]:
# df_eda = df_stat.copy()
df_eda["process_step"].value_counts()

In [ ]:
# df_eda = df_stat.copy()
df_eda = df_eda[~((df_eda["process_step"] == "confirm") & (df_eda["time_diff"] == 0))]
df_eda["process_step"].value_counts()

In [ ]:
df_eda[df_eda["client_id"] ==2182225].groupby("process_step")["time_diff"].agg("mean", "median")

In [ ]:
# df_stat.groupby('client_id').filter(functions.has_all_treatment_groups)

1. Completion rate

Test vs Control

In [ ]:
df_test = df_stat[df_stat["treatment_group"] == "Test"]
df_control = df_stat[df_stat["treatment_group"] == "Control"]

df_test["client_id"].nunique(),df_control["client_id"].nunique()

In [ ]:
df_stat

In [ ]:
df_complete_group = df_stat.groupby(['client_id', 'visitor_id', 'visit_id']).filter(functions.has_all_steps)
df_complete_group

In [ ]:
df_test_complete = df_complete_group[df_complete_group["treatment_group"] == "Test"]
df_control_complete = df_complete_group[df_complete_group["treatment_group"] == "Control"]

df_test_complete["client_id"].nunique(),df_control_complete["client_id"].nunique()

In [ ]:
# test_complete_rate = round(df_test_complete.shape[0]/df_test.shape[0],2)
# control_complete_rate = round(len(df_control_complete)/len(df_control),2)
# test_complete_rate, control_complete_rate

In [ ]:
test_complete_rate = round(df_test_complete["client_id"].nunique()/df_test["client_id"].nunique(),4)
control_complete_rate = round(df_control_complete["client_id"].nunique()/df_control["client_id"].nunique(),4)
test_complete_rate, control_complete_rate

Young vs Old

In [6]:
df_young = df_eda[df_eda["client_age_group"] == "young"]
df_old = df_eda[df_eda["client_age_group"] == "old"]

df_young["client_id"].nunique(),df_old["client_id"].nunique()

(12745, 15804)

In [7]:
df_complete_group_yo = df_eda.groupby(['client_id', 'visitor_id', 'visit_id']).filter(functions.has_all_steps)

In [ ]:
df_complete_group_yo

In [8]:
df_young_complete = df_complete_group_yo[df_complete_group_yo["client_age_group"] == "young"]
df_old_complete = df_complete_group_yo[df_complete_group_yo["client_age_group"] == "old"]

df_young_complete["client_id"].nunique(),df_old_complete["client_id"].nunique()

(9419, 10356)

In [10]:
young_complete_rate = round(df_young_complete["client_id"].nunique()/df_young["client_id"].nunique(),4)
old_complete_rate = round(df_old_complete["client_id"].nunique()/df_old["client_id"].nunique(),4)
young_complete_rate, old_complete_rate

(0.739, 0.6553)

Statistic

In [ ]:
stat,p_value = st.ttest_ind(df_test_stat[],df_control_stat[], equal_var=False, alternative="two-sided")

2. Time spent

In [ ]:
stats_df = df_stat.copy()
stats_df[((stats_df["process_step"] == "confirm") & (stats_df["time_diff"] == 0))]

In [ ]:
df_stat[df_stat["client_id"]== 1336]

In [ ]:
stats_df1 = df_complete_group.copy()
stats_df1 = stats_df1[~((stats_df1["process_step"] == "confirm") & (stats_df1["time_diff"] == 0))]
stats_df1

In [ ]:
# stats_df1 = stats_df1.groupby(['treatment_group', 'process_step'])['time_diff'].agg(["mean", "median",lambda x: list(x)]).round(2)
stats_df1 = stats_df1.groupby([ 'process_step','treatment_group',]).agg( 
    Average_Time_Diff = ('time_diff',"mean"),
    Median_Time_Diff = ('time_diff',"median"),
    Time_diff = ('time_diff',lambda x: list(x))
).round(2)

stats_df1

In [ ]:
stats_df1_treatment = stats_df1.groupby([ 'process_step']).agg( 
    Average_Time_Diff = ('time_diff',"mean"),
    Median_Time_Diff = ('time_diff',"median"),
    Time_diff = ('time_diff',lambda x: list(x))
).round(2)

stats_df1_treatment

In [ ]:
stats_df2 = df_complete_group.copy()
stats_df2 = stats_df2.groupby([ 'process_step','treatment_group']).agg( 
    Average_Time_Diff = ('time_diff',"mean"),
    Median_Time_Diff = ('time_diff',"median"),
    Time_diff = ('time_diff',lambda x: list(x))
).round(2)

stats_df2 #Without filter confirm and 0 timediff and reach confirm

In [ ]:
stats_df2_treatment = df_complete_group.copy()
stats_df2_treatment = stats_df2_treatment.groupby([ 'process_step']).agg( 
    Average_Time_Diff = ('time_diff',"mean"),
    Median_Time_Diff = ('time_diff',"median"),
    Time_diff = ('time_diff',lambda x: list(x))
).round(2)

stats_df2_treatment 

In [ ]:
stats_df3 = df_stat.copy()
stats_df3 = stats_df3.groupby([ 'process_step','treatment_group']).agg( 
    Average_Time_Diff = ('time_diff',"mean"),
    Median_Time_Diff = ('time_diff',"median"),
    Time_diff = ('time_diff',lambda x: list(x))
).round(2)

stats_df3

In [ ]:
stats_df3_treament = df_stat.copy()
stats_df3_treament = stats_df3_treament.groupby([ 'process_step']).agg( 
    Average_Time_Diff = ('time_diff',"mean"),
    Median_Time_Diff = ('time_diff',"median"),
    Time_diff = ('time_diff',lambda x: list(x))
).round(2)

stats_df3_treament

In [ ]:
stats_df4_treament = df_stat.copy()
stats_df4_treament = stats_df4_treament[~((stats_df4_treament["process_step"] == "confirm") & (stats_df4_treament["time_diff"] == 0))]

stats_df4_treament = stats_df4_treament.groupby([ 'process_step','treatment_group']).agg( 
    Average_Time_Diff = ('time_diff',"mean"),
    Median_Time_Diff = ('time_diff',"median"),
    Time_diff = ('time_diff',lambda x: list(x))
).round(2)

stats_df4_treament

In [13]:
stats_df5_treament = df_stat.copy()
stats_df5_treament = stats_df5_treament[~((stats_df5_treament["process_step"] == "confirm") & (stats_df5_treament["time_diff"] == 0))]
stats_df5_treament = stats_df5_treament[stats_df5_treament["time_diff"] < 1000]

stats_df5_treament = stats_df5_treament.groupby([ 'process_step','treatment_group']).agg( 
    Average_Time_Diff = ('time_diff',"mean"),
    Median_Time_Diff = ('time_diff',"median"),
    Time_diff = ('time_diff',lambda x: list(x))
).round(2)

stats_df5_treament

Average_Time_Diff  Median_Time_Diff  \
process_step treatment_group                                        
confirm      Control                     317.17             260.0   
             Test                        293.82             229.0   
start        Control                      78.50               0.0   
             Test                         95.38               0.0   
step_1       Control                      96.05              29.0   
             Test                        105.78              19.0   
step_2       Control                     136.34              71.0   
             Test                        148.40              69.0   
step_3       Control                     223.96             166.0   
             Test                        223.29             158.0   

                                                                      Time_diff  
process_step treatment_group                                                     
confirm      Control          [245.0, 95.0, 292.0, 90.0, 443.0, 227.0, 626.0...  
             Test             [158.0, 377.0, 211.0, 82.0, 954.0, 688.0, 223....  
start        Control          [0.0, 0.0, 0.0, 0.0, 0.0, 804.0, 0.0, 0.0, 0.0...  
             Test             [0.0, 0.0, 0.0, 32.0, 114.0, 142.0, 0.0, 0.0, ...  
step_1       Control          [49.0, 112.0, 507.0, 527.0, 538.0, 11.0, 33.0,...  
             Test             [7.0, 7.0, 25.0, 8.0, 206.0, 5.0, 112.0, 245.0...  
step_2       Control          [121.0, 529.0, 22.0, 131.0, 10.0, 72.0, 120.0,...  
             Test             [39.0, 25.0, 51.0, 213.0, 68.0, 149.0, 237.0, ...  
step_3       Control          [396.0, 162.0, 61.0, 78.0, 574.0, 193.0, 38.0,...  
             Test             [138.0, 214.0, 102.0, 262.0, 495.0, 698.0, 658...

In [11]:
stats_df5_treament_yo = df_eda.copy()
stats_df5_treament_yo

,client_id,visitor_id,visit_id,process_step,date_time,treatment_group,time_diff,time_diff_relative,time_diff_percent,client_tenure_year,client_tenure_month,client_age,gender,num_accts,balance,calls_6_mnth,logons_6_mnth,client_age_group
0,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test,0.0,0.0,0.00,3,46,29,U,2,25454.66,2,6,young
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test,7.0,7.0,0.04,3,46,29,U,2,25454.66,2,6,young
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test,39.0,32.0,0.25,3,46,29,U,2,25454.66,2,6,young
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test,138.0,99.0,0.87,3,46,29,U,2,25454.66,2,6,young
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test,158.0,20.0,1.00,3,46,29,U,2,25454.66,2,6,young
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305734,9999729,834634258_21862004160,870243567_56915814033_814203,step_2,2017-05-08 16:08:40,Test,15.0,10.0,0.20,10,124,31,F,3,107059.74,6,9,young
305735,9999729,834634258_21862004160,870243567_56915814033_814203,step_3,2017-05-08 16:09:19,Test,54.0,39.0,0.72,10,124,31,F,3,107059.74,6,9,young
305736,9999729,834634258_21862004160,870243567_56915814033_814203,confirm,2017-05-08 16:09:40,Test,75.0,21.0,1.00,10,124,31,F,3,107059.74,6,9,young
305737,9999832,145538019_54444341400,472154369_16714624241_585315,start,2017-05-16 16:46:03,Test,0.0,0.0,0.00,23,281,49,F,2,431887.61,1,4,middle-aged


In [12]:

stats_df5_treament_yo = stats_df5_treament_yo[~((stats_df5_treament_yo["process_step"] == "confirm") & (stats_df5_treament_yo["time_diff"] == 0))]
stats_df5_treament_yo = stats_df5_treament_yo[stats_df5_treament_yo["time_diff"] < 1000]

stats_df5_treament_yo = stats_df5_treament_yo.groupby([ 'process_step','client_age_group']).agg( 
    Average_Time_Diff = ('time_diff',"mean"),
    Median_Time_Diff = ('time_diff',"median"),
    Time_diff = ('time_diff',lambda x: list(x))
).round(2)

stats_df5_treament_yo

Average_Time_Diff  Median_Time_Diff  \
process_step client_age_group                                        
confirm      middle-aged                  303.10             246.0   
             old                          349.88             290.0   
             young                        252.49             194.0   
start        middle-aged                   85.47               0.0   
             old                          104.42               0.0   
             young                         66.76               0.0   
step_1       middle-aged                   95.70              21.0   
             old                          125.06              40.0   
             young                         76.10              15.0   
step_2       middle-aged                  134.62              64.0   
             old                          178.04              99.0   
             young                        106.23              44.0   
step_3       middle-aged                  229.48             171.0   
             old                          241.89             175.0   
             young                        189.84             135.0   

                                                                       Time_diff  
process_step client_age_group                                                     
confirm      middle-aged       [245.0, 211.0, 82.0, 688.0, 223.0, 117.0, 309....  
             old               [377.0, 954.0, 632.0, 183.0, 125.0, 626.0, 954...  
             young             [158.0, 95.0, 208.0, 292.0, 412.0, 315.0, 141....  
start        middle-aged       [0.0, 32.0, 114.0, 142.0, 0.0, 0.0, 0.0, 0.0, ...  
             old               [0.0, 0.0, 162.0, 200.0, 213.0, 231.0, 0.0, 0....  
             young             [0.0, 0.0, 0.0, 0.0, 20.0, 21.0, 97.0, 0.0, 15...  
step_1       middle-aged       [49.0, 112.0, 507.0, 527.0, 538.0, 33.0, 25.0,...  
             old               [7.0, 5.0, 112.0, 245.0, 56.0, 14.0, 11.0, 17....  
             young             [7.0, 11.0, 4.0, 105.0, 13.0, 5.0, 251.0, 16.0...  
step_2       middle-aged       [121.0, 529.0, 131.0, 51.0, 213.0, 76.0, 57.0,...  
             old               [25.0, 68.0, 149.0, 237.0, 91.0, 120.0, 37.0, ...  
             young             [39.0, 22.0, 10.0, 72.0, 118.0, 180.0, 52.0, 2...  
step_3       middle-aged       [396.0, 162.0, 102.0, 262.0, 495.0, 658.0, 96....  
             old               [214.0, 698.0, 574.0, 84.0, 264.0, 81.0, 207.0...  
             young             [138.0, 61.0, 78.0, 178.0, 191.0, 193.0, 389.0...

3. Error rate

In [ ]:
df_error = df_stat.groupby(['client_id', 'visit_id', 'visitor_id']).apply(functions.calculate_error_rate).reset_index()

# Calculate avg error rate by treatment_group
avg_error_rates = df_error.groupby('treatment_group')['error_rate'].mean().reset_index()
avg_error_rates